<a href="https://colab.research.google.com/github/niya007-ui/movie-recommendation-system/blob/main/movierecomd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommender: EDA and Popularity Baseline

Exploring the MovieLens dataset and building a baseline that ranks movies by a weighted rating.

In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-11-24 10:10:34--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  4.37MB/s    in 0.2s    

2026-11-24 10:10:34 (4.37 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [3]:
import pandas as pd

ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [4]:
# Join ratings with movie titles
merged = ratings.merge(movies, on='movieId')

# One row per movie: number of ratings and average rating
movie_stats = merged.groupby('title')['rating'].agg(['count', 'mean'])

# Weighted rating: pull movies with few ratings toward the global average
global_mean = ratings['rating'].mean()
min_ratings = 50

movie_stats['weighted'] = (
    movie_stats['count'] / (movie_stats['count'] + min_ratings) * movie_stats['mean']
    + min_ratings / (movie_stats['count'] + min_ratings) * global_mean
)

top10 = movie_stats.sort_values('weighted', ascending=False).head(10).round(2)
top10

,count,mean,weighted
title,,,
"Shawshank Redemption, The (1994)",317,4.43,4.30
Fight Club (1999),218,4.27,4.13
"Godfather, The (1972)",192,4.29,4.13
Star Wars: Episode IV - A New Hope (1977),251,4.23,4.11
Pulp Fiction (1994),307,4.20,4.10
"Usual Suspects, The (1995)",204,4.24,4.09
Schindler's List (1993),220,4.22,4.09
"Matrix, The (1999)",278,4.19,4.09
Star Wars: Episode V - The Empire Strikes Back (1980),211,4.22,4.08


In [5]:
print('Ratings:', len(ratings))
print('Users:', ratings['userId'].nunique())
print('Movies rated:', ratings['movieId'].nunique())
print('Most common rating:', ratings['rating'].value_counts().idxmax())

Ratings: 100836
Users: 610
Movies rated: 9724
Most common rating: 4.0
